<a href="https://colab.research.google.com/github/saffarizadeh/INSY5378/blob/main/Assignments/Assignment_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://kambizsaffari.com/Logo/College_of_Business.cmyk-hz-lg.png" width="500px"/>

# *INSY 5378*

# **Assignment: Stock Price Prediction with RNNs**

Instructor: Dr. Kambiz Saffari

---

**Instructions**
- This assignment is based on **Chapter 13: Timeseries Forecasting**. You are expected to have read the chapter and gone through the class notebook before attempting this assignment.
- The chapter is available online at: https://deeplearningwithpython.io/chapters/chapter13_timeseries-forecasting/
- Each question points you to the relevant concepts from the chapter. **Review those sections before attempting the question.**
- Write your answers directly in the provided cells.
- You may add additional cells if you want to test ideas, but only answers in the marked cells will be graded.

**Disclaimer:** The predictions produced in this assignment are strictly for educational purposes. They should not be used to make actual trading or investment decisions.

## Setup

Run the cells below to install dependencies and configure the backend.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"

In [ ]:
!pip install keras keras-hub yfinance --upgrade -q

## Question 1: Data Acquisition and Preparation

**Review:** The sections titled **"A temperature forecasting example"** and **"Preparing the data"** in Chapter 13. Pay close attention to how the chapter handles normalization, chronological splitting, and the use of `timeseries_dataset_from_array()`.

---

**Part A: Download and Explore**

1. Use `yfinance` to download **10 years** of daily historical data for a stock of your choice (e.g., `"AAPL"`, `"MSFT"`, `"GOOGL"`). Store the result in a DataFrame called `company_data`.
2. Display the first few rows with `.head()` and print the shape.
3. Plot the **Close** price over time.
4. In a comment, describe any visible patterns (trends, periodicity, volatility changes).

**Part B: Normalize and Split**

1. Select the following columns as your features: `"Open"`, `"High"`, `"Low"`, `"Close"`, `"Volume"`. Store them as a NumPy array called `raw_data`. Also extract the `"Close"` column separately into a NumPy array called `close_prices` (this will be your prediction target).
2. Split the data **chronologically** (not randomly): first 70% for training, next 15% for validation, last 15% for testing. Store the counts in `num_train_samples`, `num_val_samples`, and `num_test_samples`.
3. Compute the mean and standard deviation of `raw_data` **using only the training portion**, then normalize all of `raw_data`. In a comment, explain why you must not use validation/test data to compute these statistics (refer to the chapter's discussion of this point).

**Part C: Create Sequence Datasets**

Use `sequence_length=30` (the model sees 30 trading days of history) and predict the closing price **1 day ahead**.

1. Compute the appropriate `delay` value for a 1-day-ahead prediction (see how the chapter computes `delay` in the Jena example, but note that here our `sampling_rate` is 1 since the data is already daily).
2. Using `keras.utils.timeseries_dataset_from_array()`, create `train_dataset`, `val_dataset`, and `test_dataset` with `batch_size=32`. Use the appropriate `start_index` and `end_index` to respect your chronological split.
3. Print the shape of one batch of samples and targets from `train_dataset` and, in a comment, explain what each dimension represents.

In [ ]:
# Write your answer to Question 1 here

## Question 2: RNN Models for Stock Prediction

**Review:** The sections titled **"A commonsense, non-machine-learning baseline"**, **"Recurrent neural networks"**, **"Using recurrent dropout to fight overfitting"**, and **"Stacking recurrent layers"** in Chapter 13.

---

**Part A: Commonsense Baseline and Simple LSTM**

The chapter emphasizes that you should always establish a **commonsense baseline** before trying ML models. For stock prices, a reasonable heuristic is: *predict that tomorrow's closing price equals today's closing price*.

1. Write a function `evaluate_naive_method(dataset)` that implements this baseline. For each batch, use the last closing price in the input sequence as the prediction. Since the data is normalized, **un-normalize** predictions before computing the error. Print the **Mean Absolute Error (MAE)** in dollars on the validation and test sets.
   *Hint: the Close price is at column index 3 in `raw_data`. To un-normalize, multiply by `std[3]` and add `mean[3]`.*
2. Build a model with a single `LSTM(32)` layer followed by `Dense(1)`. Compile with `loss="mse"` and `metrics=["mae"]`. Train for **20 epochs** using `ModelCheckpoint` to save the best model.
3. Plot the training and validation MAE over epochs.
4. Load the best model and evaluate on the test set. In a comment, compare the LSTM's test MAE to the baseline. Did the LSTM beat it?

**Part B: LSTM with Recurrent Dropout**

1. Build a model with `LSTM(64, recurrent_dropout=0.2)`, followed by `Dropout(0.3)`, followed by `Dense(1)`. Train for **50 epochs** with `ModelCheckpoint`.
2. Plot training and validation MAE.
3. Load the best model and evaluate on the test set. In a comment, compare the training and validation curves to Part A. Does recurrent dropout reduce overfitting? Reference the chapter's explanation of why the same dropout mask must be applied at every timestep (instead of a random mask that changes each timestep).

**Part C: Stacked GRU**

1. Build a model with two `GRU` layers (32 units each), both using `recurrent_dropout=0.2`. The first must use `return_sequences=True`. Add `Dropout(0.3)` before the final `Dense(1)`. Train for **50 epochs** with `ModelCheckpoint`.
2. Plot training and validation MAE.
3. Load the best model and evaluate on the test set. In a comment, explain why `return_sequences=True` is required on the first GRU layer but not the second (reference the chapter's discussion of stacking recurrent layers).

**Part D: Model Comparison**

1. Create a bar chart comparing the **test MAE (in dollars)** of all four approaches: commonsense baseline, simple LSTM, LSTM with dropout, and stacked GRU.
2. In a comment, discuss:
   - Which model performed best, and by how much did it improve over the baseline?
   - Weather data has strong physical periodicity (daily and yearly cycles) that make it predictable. How does stock data differ?
   - Based on your results, are RNNs effective for stock price prediction? Why or why not?

In [ ]:
# Write your answer to Question 2 here

---
**Submission Reminder**
- Make sure your notebook runs from top to bottom without errors.
- Clearly label all answers.
- Include all required comments and explanations. They are part of your grade.